In [ ]:
# Setup — all imports live here so the notebook executes top-to-bottom
# without reimporting mid-notebook. When running headlessly (CI, HPC),
# uncomment the ``matplotlib.use('Agg')`` line *before* the pyplot import
# so figures never need an interactive display.
import dataclasses
import logging
import os
import sys
from collections import Counter
from pathlib import Path

# import matplotlib
# matplotlib.use('Agg')  # enable for headless runs
import matplotlib.pyplot as plt
import mne
import numpy as np
import seaborn as sns
from matplotlib.colors import BoundaryNorm, ListedColormap
from mne.viz import plot_topomap
from scipy.sparse.csgraph import connected_components
from scipy.stats import pearsonr

# Resolve project root regardless of where the notebook is launched from
for _candidate in [".", "..", "../.."]:  # noqa: B007
    _p = os.path.abspath(_candidate)
    if os.path.isdir(os.path.join(_p, "src")):
        sys.path.insert(0, _p)
        break

from independent_vector_analysis import iva_g  # noqa: E402
from sklearn.decomposition import PCA  # noqa: E402

from scripts.notebook_helpers import (  # noqa: E402
    WAVELET_FREQ_MAX,
    WAVELET_FREQ_MIN,
    WAVELET_N_FREQS,
    analyzers_to_datasets,
    compute_wavelet_datasets,
    load_analyzers,
)
from src.analysis.wavelet_ica import (  # noqa: E402
    align_iva_component_signs,
    iva_component_patterns,
    normalize_patterns_per_subject,
    zscore_by_time,
)
from src.definitions.constants import ProjectPaths  # noqa: E402
from src.definitions.fields import (  # noqa: E402
    ConditionVariants,
    ExclusionCategories,
    ExperimentNames,
    MusicTypeVariants,
)

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.0)
%matplotlib inline

# IVA Decomposition of Wavelet Power — Frequency × Channel Components

## Naming Convention

Notebooks in this directory are named after the **independent
dimension** — the feature axis where the IVA components live. Here
the per-subject feature axis is `(frequency × channel)`, so the
components are spatio-spectral patterns over `(F, C)`. The K-axis
(the *dataset* axis IVA aligns scores across) is the **subject**
axis.

We use **`scores`** (rather than the more common IVA term *sources*)
for the per-observation weights `S = W @ X` to stay consistent with
the ICA notebooks in [`04-wavelet-ica-analysis/`](../04-wavelet-ica-analysis/),
where the analogous variables are named `ica_scores`, `pca_scores`,
`scores_3d`.

## Scope

Run the **Independent Vector Analysis** algorithm on the 4-D wavelet
power tensor and stop. Analyses of the resulting scores / mixing
matrices live in companion notebooks (next steps).

Unlike single-dataset ICA, **IVA decomposes K datasets jointly** while
preserving the dependency between corresponding scores across
datasets. Here each **subject is one dataset**, so IVA yields a set
of components that are *already aligned* across subjects — the kth
score in subject 1 corresponds to the kth score in subjects 2..K.
This removes the permutation ambiguity that single-subject ICA
introduces and is the main motivation for choosing IVA over ICA in a
group analysis.

## Reshape

Each subject's wavelet tensor is flattened so that **channels ×
frequencies** form the observation axis and **time** is the sample
axis:

```
Input:   (n_subjects, n_channels, n_freqs, n_times)
Per-subject reshape:
         (n_channels, n_freqs, n_times)
         → (n_channels × n_freqs,  n_times)
           ──── features ──────   samples
Stack:   (n_pca,  n_times,  n_subjects)  — IVA input layout (N, T, K)
```

IVA-G requires a **square** mixing matrix per dataset (`N == feature
dim`), so each subject's `(C×F, T)` matrix is first reduced with a
**per-subject PCA** to `N_PCA` components before stacking.

## Pipeline

1. Load wavelet power for the chosen condition / music type (cached).
2. Apply subject / channel / time subsets for fast iteration.
3. Z-score along time.
4. Per-subject reshape to `(C×F, T)` and per-subject PCA to `N_PCA`.
5. Stack into `(N_PCA, T, K)` and run **IVA-G**.
6. Recover per-subject scores and per-subject component patterns in
   the original `(F, C)` feature space.

After the final cell the following variables are available for any
downstream analysis notebook:

| Variable | Shape | Description |
|----------|-------|-------------|
| `bb_data` | `(S, C, F, T)` | Raw 4-D wavelet power tensor |
| `bb_z` | `(S, C, F, T)` | Z-scored 4-D wavelet power tensor |
| `X_subjects` | `(S, C×F, T)` | Per-subject z-scored reshape |
| `pcas` | list[`PCA`] | Per-subject fitted PCA objects |
| `X_pca` | `(N_PCA, T, S)` | PCA-reduced IVA input layout |
| `W` | `(N_PCA, N_PCA, S)` | Per-subject IVA demixing matrices |
| `cost` | `(n_iter,)` | IVA cost per iteration |
| `iva_scores_pca` | `(S, N_PCA, T)` | Per-subject IVA scores in PCA space |
| `iva_components` | `(S, N_PCA, F, C)` | IVA component patterns reshaped to (freq, channel) per subject |

## Configuration

In [ ]:
# ── Experiment configuration ───────────────────────────────────
# Choose the experiment: ExperimentNames.PSILO_MUSIC or ExperimentNames.ASSR
EXPERIMENT_NAME = ExperimentNames.PSILO_MUSIC
CONDITION = ConditionVariants.PLACEBO
if EXPERIMENT_NAME == ExperimentNames.ASSR:
    # ASSR has no music dimension; uses a single placeholder "music type".
    MUSIC_TYPES = [MusicTypeVariants.ASSR]
else:
    MUSIC_TYPES = [MusicTypeVariants.CLASSICAL]  # single type for fast exploration
EXCLUSION_CATEGORIES = [ExclusionCategories.BAD_MUSIC, ExclusionCategories.ARTIFACTS]
PROCESS_AND_SAVE_DATA = False  # set True to re-process raw files

# ── Wavelet settings ─────────────────────────────────────────
REPRESENTATION = "power"
# Wavelet frequency grid: canonical 1 Hz-spaced grid from src.definitions.frequency
# (WAVELET_FREQ_MIN/MAX/N_FREQS), re-exported via scripts.notebook_helpers.
FREQS = np.linspace(WAVELET_FREQ_MIN, WAVELET_FREQ_MAX, WAVELET_N_FREQS)

KEEP_FREQUENCY_DIM = True
RESHAPE_FREQUENCY_DIM = True  # -> (n_subjects, n_channels, n_freqs, n_times)

# ── Reuse / compute ──────────────────────────────────────────
REUSE_WAVELETS = True  # load from cache; set False to compute + save

# ── Subject subset ────────────────────────────────────────────
N_SUBJECTS_SUBSET: int | None = 5

# ── Channel and time subset ─────────────────────────────────────
N_CHANNELS_SUBSET: int | None = 32  # first N channels (from 195)
N_TIMES_SUBSET: int | None = 10000  # first N time samples

# ── IVA settings ──────────────────────────────────────────────
# N_COMPONENTS_PCA must be ≥ N_TOP + N_BOTTOM (below) for the
# ranking step to have something to choose from. For a real run on
# 257-channel data target ~50 PCs (≈0.8 EVR); 30 is a reasonable
# default for fast exploration.
N_COMPONENTS_PCA = 30  # per-subject PCA dim before IVA (= N in IVA's (N, T, K))
IVA_OPT_APPROACH = "newton"  # 'gradient', 'newton', or 'quasi'
IVA_MAX_ITER = 64
IVA_W_DIFF_STOP = 1e-6
IVA_VERBOSE = True
IVA_RANDOM_STATE = 42  # seeds per-subject PCA + W_init

# ── Storage directory ─────────────────────────────────────────
# Reuse the shared stage-03 wavelet cache so the Morlet transform is computed
# once (in notebooks/03-wavelet-analysis) and never recomputed in 04/05.
WAVELET_DIR: Path = (
    ProjectPaths.NOTEBOOKS_DIR
    / "03-wavelet-analysis"
    / "wavelet_cache"
    / EXPERIMENT_NAME.value
)

# ── Downstream analysis settings ──────────────────────────────
# Components are ranked by mean LOO-ISC on the score timecourses
# `iva_scores_pca` (the IVA-aligned sample axis — time, in this
# notebook). Downstream cells visualise the TOP and BOTTOM
# components separately so the bottom set serves as a noise /
# no-alignment contrast.
N_TOP = 10
N_BOTTOM = 5
N_COMPONENTS_SHOW = N_TOP + N_BOTTOM  # total panels per downstream analysis
SAVE_PLOTS = True
# Plots directory follows the new naming convention: the independent
# dimension (where the components live) is `(frequency × channel)`.
PLOTS_DIR = (
    ProjectPaths.NOTEBOOKS_DIR
    / "05-wavelet-iva-analysis"
    / "plots"
    / EXPERIMENT_NAME.value
    / "broadband"
    / "iva_frequency_channel"
    / f"pca_{N_COMPONENTS_PCA}"  # isolate sweeps over N_COMPONENTS_PCA
)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Wavelet base directory : {WAVELET_DIR}")
print(f"Plots directory        : {PLOTS_DIR}")
print(
    f"Frequencies            : {FREQS[0]:.1f}–{FREQS[-1]:.1f} Hz ({len(FREQS)} steps)"
)
print(f"Per-subject PCA dim    : {N_COMPONENTS_PCA}")
print(f"IVA optimisation       : {IVA_OPT_APPROACH}  (max_iter={IVA_MAX_ITER})")
print(
    f"Components to show     : top={N_TOP} + bottom={N_BOTTOM} "
    f"(total {N_COMPONENTS_SHOW})"
)


## Data Loading

In [ ]:
analyzers = load_analyzers(
    MUSIC_TYPES,
    CONDITION,
    EXCLUSION_CATEGORIES,
    PROCESS_AND_SAVE_DATA,
    normalize_data=False,
    experiment_name=EXPERIMENT_NAME,
)
datasets = analyzers_to_datasets(analyzers)

# Always limit to first N_SUBJECTS_SUBSET individuals
if N_SUBJECTS_SUBSET is not None:
    datasets = {
        label: dataclasses.replace(ad, data=ad.data[:N_SUBJECTS_SUBSET])
        for label, ad in datasets.items()
    }
    print(f"Using first {N_SUBJECTS_SUBSET} individuals.")

# Slice to channel subset
if N_CHANNELS_SUBSET is not None:
    datasets = {
        label: dataclasses.replace(ad, data=ad.data[:, :N_CHANNELS_SUBSET, :])
        for label, ad in datasets.items()
    }
    print(f"Using first {N_CHANNELS_SUBSET} channels.")

# Slice to time subset
if N_TIMES_SUBSET is not None:
    datasets = {
        label: dataclasses.replace(ad, data=ad.data[:, :, :N_TIMES_SUBSET])
        for label, ad in datasets.items()
    }
    print(f"Using first {N_TIMES_SUBSET} time samples.")

print("Loaded datasets:", list(datasets.keys()))
for label, ad in datasets.items():
    print(
        f"  {label}: {ad.n_items} subjects, {ad.n_features} channels, "
        f"{ad.n_samples} samples"
    )

## Load or Compute Wavelet Transforms

Stored in `WAVELET_DIR/broadband/` (shared with the 04 ICA notebooks).

In [ ]:
broadband_datasets = compute_wavelet_datasets(
    datasets=datasets,
    analyzers=analyzers,
    experiment_name=EXPERIMENT_NAME,
    freqs=FREQS,
    representation=REPRESENTATION,
    keep_frequency_dim=KEEP_FREQUENCY_DIM,
    reshape_frequency_dim=RESHAPE_FREQUENCY_DIM,
    wavelet_dir=WAVELET_DIR / "broadband",
    reuse_wavelets=REUSE_WAVELETS,
)
for label, ad in broadband_datasets.items():
    source = ad.metadata.get("loaded_from_wavelet_file", "computed_now")
    print(f"[broadband] {label}: shape={ad.data.shape}  source={source}")

## Dataset Selection

Change `LABEL` to switch between music types. The remaining cells use
`bb_data` (broadband wavelet power, 4-D) and derived quantities.

In [ ]:
LABEL = list(broadband_datasets.keys())[0]

bb_ad = broadband_datasets[LABEL]
bb_data = bb_ad.data  # (n_subjects, n_channels, n_freqs, n_times)
sfreq = bb_ad.sfreq

n_subjects, n_channels, n_freqs, n_times = bb_data.shape
time = np.arange(n_times) / sfreq

print(f"Dataset    : {LABEL}")
print(f"Shape      : {bb_data.shape}  (subjects × channels × freqs × times)")
print(f"Duration   : {n_times / sfreq:.1f} s  @  {sfreq} Hz")
print(f"Freq range : {FREQS[0]:.1f}–{FREQS[-1]:.1f} Hz ({n_freqs} steps)")

---
## Step 1 — Z-score and Per-Subject Reshape

**Z-scoring** normalises each `(subject, channel, frequency)` time
series to zero mean and unit variance so that IVA is not dominated by
high-power channels, subjects, or frequencies.

**Reshape** is done independently per subject: for each subject `k`
the `(C, F, T)` slice becomes a `(C×F, T)` matrix — channels and
frequencies form the observation axis, time stays as samples. The
result is stacked as a `(S, C×F, T)` array (Python view; IVA wants
axis-ordering `(N, T, K)` which we build in the next step).

| Quantity | Shape | Description |
|----------|-------|-------------|
| `bb_z` | `(S, C, F, T)` | Z-scored wavelet power |
| `X_subjects` | `(S, C×F, T)` | Per-subject z-scored reshape |

In [ ]:
# Z-score along time: each (subject, channel, frequency) slice → mean=0, std=1
bb_z = zscore_by_time(bb_data)  # (S, C, F, T)

# Per-subject reshape: (S, C, F, T) → (S, C*F, T)
n_feat = n_channels * n_freqs
X_subjects = bb_z.reshape(n_subjects, n_feat, n_times)  # (S, C*F, T)

print(f"Per-subject reshape    : {X_subjects.shape}  (subjects, C*F, time)")
print(f"  Features per subject : {n_feat}  (C={n_channels} * F={n_freqs})")
print(f"  Samples per subject  : {n_times}")

---
## Step 2 — Per-Subject PCA Dimensionality Reduction

`iva_g` assumes a **square** mixing matrix per dataset, i.e. the
number of estimated scores equals the input dimensionality. Running
IVA directly on `(C×F)` features per subject would be prohibitively
expensive (and rank-deficient if `C×F > T`). Each subject's matrix
is reduced with its **own** PCA to `N_PCA` components, after which the
K subject matrices are stacked into IVA's `(N, T, K)` layout.

Keeping per-subject PCAs (rather than a single shared PCA) preserves
subject-specific spatial / spectral subspaces, which is exactly what
IVA exploits to align cross-subject scores.

| Quantity | Shape | Description |
|----------|-------|-------------|
| `pcas[k]` | — | Fitted PCA object for subject `k` |
| `pca_evr` | `(S, N_PCA)` | Explained-variance ratio per subject |
| `X_pca` | `(N_PCA, T, S)` | IVA input layout (N, T, K) |

In [ ]:
# Per-subject PCA. PCA expects (n_samples, n_features); each subject's matrix
# is (C*F, T) → transpose to (T, C*F), fit, transform back to (T, N_PCA),
# transpose to (N_PCA, T), then stack along axis 2 as IVA expects.
pcas: list[PCA] = []
pca_scores_per_subject = np.zeros((n_subjects, N_COMPONENTS_PCA, n_times))
pca_evr = np.zeros((n_subjects, N_COMPONENTS_PCA))

for k in range(n_subjects):
    subj_matrix = X_subjects[k].T  # (T, C*F) — samples x features for sklearn
    pca = PCA(n_components=N_COMPONENTS_PCA, random_state=IVA_RANDOM_STATE)
    scores = pca.fit_transform(subj_matrix)  # (T, N_PCA)
    pcas.append(pca)
    pca_scores_per_subject[k] = scores.T  # (N_PCA, T)
    pca_evr[k] = pca.explained_variance_ratio_
    print(
        f"  S{k + 1}: explained variance = {pca_evr[k].sum() * 100:5.1f}% "
        f"({N_COMPONENTS_PCA} comps)"
    )

# IVA expects (N, T, K)
X_pca = np.ascontiguousarray(pca_scores_per_subject.transpose(1, 2, 0))

print(f"\nIVA input shape        : {X_pca.shape}  (N_PCA, T, K=subjects)")

# Quick visual of how much variance each subject's PCA retains.
fig, ax = plt.subplots(figsize=(8, 4))
for k in range(n_subjects):
    ax.plot(
        np.arange(1, N_COMPONENTS_PCA + 1),
        np.cumsum(pca_evr[k]),
        marker="o",
        markersize=3,
        label=f"S{k + 1}",
    )
ax.set_xlabel("Number of PCA components")
ax.set_ylabel("Cumulative variance explained")
ax.set_title(f"Per-Subject PCA — {LABEL}")
ax.legend(fontsize=8, ncol=2)
ax.axhline(0.9, ls="--", lw=0.6, color="gray")
fig.tight_layout()
plt.show()
plt.close("all")

---
## Step 3 — Run IVA-G

`iva_g` returns a demixing matrix `W` of shape `(N, N, K)`. The
scores for subject `k` are then

```
S_pca[k]  = W[:, :, k] @ X_pca[:, :, k]                 # (N_PCA, T)
patterns[k] = iva_component_patterns(W[:, :, k], pcas[k].components_)  # (N_PCA, C*F)
S_full[k] = patterns[k].T @ S_pca[k]                     # (C*F, T)
components[k] = patterns[k]
             .reshape(N_PCA, F, C)
```

Because IVA's permutation ambiguity is **shared across datasets**, the
kth score in `S_full[0]` corresponds to the kth score in
`S_full[1..K-1]` — there is no need to align scores across subjects
post-hoc.

**Topographies are the forward (mixing) patterns**, i.e.
`pinv(W_k @ pcas[k].components_)` — *not* the unmixing rows. `iva_g` folds its
internal whitening `V_k` into the returned `W_k`, so the unmixing rows carry an
extra `Σ⁻¹` weighting that up-weights the lowest-variance retained PCA
directions; the filter and the pattern of the same component can be nearly
uncorrelated. `iva_component_patterns` does the inversion (same convention as
MNE's `ica.get_components()`).

| Returned by `iva_g` | Shape | Description |
|---------------------|-------|-------------|
| `W` | `(N_PCA, N_PCA, S)` | Per-subject demixing matrix |
| `cost` | `(n_iter,)` | IVA cost per iteration |
| `Sigma_N` | `(S, S, N_PCA)` | Per-score-component covariance across subjects |
| `isi` | `float` | Joint inter-symbol-interference (only when ground-truth `A` is given) |

In [ ]:
# Deterministic W initialisation: random matrix per subject seeded by IVA_RANDOM_STATE.
rng = np.random.default_rng(IVA_RANDOM_STATE)
W_init = rng.standard_normal((N_COMPONENTS_PCA, N_COMPONENTS_PCA, n_subjects))

W, cost, Sigma_N, isi = iva_g(
    X_pca,
    opt_approach=IVA_OPT_APPROACH,
    whiten=True,
    verbose=IVA_VERBOSE,
    W_init=W_init,
    max_iter=IVA_MAX_ITER,
    W_diff_stop=IVA_W_DIFF_STOP,
)

print(f"\nW shape           : {W.shape}  (N_PCA, N_PCA, K=subjects)")
print(f"Sigma_N shape     : {Sigma_N.shape}  (K, K, N_PCA)")
print(f"Iterations        : {len(cost)}")
print(f"Final cost        : {cost[-1]:.6f}")

# Cost-curve sanity check.
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(cost, marker="o", markersize=3, color="steelblue")
ax.set_xlabel("Iteration")
ax.set_ylabel("IVA cost")
ax.set_title(f"IVA-G Convergence — {LABEL}")
fig.tight_layout()
plt.show()
plt.close("all")

In [ ]:
np.diag(Sigma_N[:, :, 0])

In [ ]:
# Resolve IVA's per-subject sign ambiguity. IVA recovers each component (SCV)
# only up to a per-subject sign, so subject i's copy of component k may be the
# negation of subject j's — which makes genuinely shared activity look
# anti-correlated. For each component we normalise Sigma_N to a subject x subject
# correlation matrix, take its leading eigenvector (the dominant cross-subject
# direction, oriented so its largest-magnitude entry is positive), and flip every
# subject whose loading on it is negative. Flips are applied to both W and the
# correlation stack (sigma_corr), so every quantity recovered from W below is
# sign-aligned across subjects.
sigma_corr, W, sign_flips = align_iva_component_signs(Sigma_N, W)
n_flipped = int((sign_flips < 0).sum())
print(
    f"Sign alignment: flipped {n_flipped} (component, subject) pairs "
    f"across {N_COMPONENTS_PCA} components."
)
print(f"sigma_corr shape  : {sigma_corr.shape}  (N_PCA, K, K)")
print(f"W shape           : {W.shape}  (N_PCA, N_PCA, K) — sign-aligned")

---
## Step 4 — Recover Scores and Component Patterns

Apply the per-subject demixing matrix to the PCA-reduced data to
obtain IVA scores, then combine `W_k` with the PCA loadings to
express each component pattern over the original `(F, C)` axes
(directly comparable to `components_2d` in the 04 ICA notebook).

The kth row of `iva_components[k]` is the unmixing vector for the
kth score in subject `k`'s original feature space. Because IVA
resolves the permutation jointly across datasets, the kth row is
**already aligned** across subjects.

In [ ]:
iva_scores_pca = np.zeros((n_subjects, N_COMPONENTS_PCA, n_times))
iva_components = np.zeros((n_subjects, N_COMPONENTS_PCA, n_freqs, n_channels))

for k in range(n_subjects):
    W_k = W[:, :, k]  # (N_PCA, N_PCA)
    X_pca_k = X_pca[:, :, k]  # (N_PCA, T)

    # Scores in the PCA subspace: (N_PCA, T)
    iva_scores_pca[k] = W_k @ X_pca_k

    # Forward (mixing) patterns in the full (C*F) feature space, NOT
    # the unmixing rows — after iva_g's internal whitening those carry
    # a Σ⁻¹ reweighting of the PCA directions.
    components_full = iva_component_patterns(
        W_k, pcas[k].components_
    )  # (N_PCA, C*F)

    # X_subjects was reshape (S, C, F, T) → (S, C*F, T), so the flattened axis
    # iterates channels (slow) then frequencies (fast). Reshape back to
    # (N_PCA, C, F) and transpose to (N_PCA, F, C) to mirror the 04 ICA
    # notebook's components_2d layout.
    iva_components[k] = components_full.reshape(
        N_COMPONENTS_PCA, n_channels, n_freqs
    ).transpose(0, 2, 1)

print(f"IVA scores (PCA space)   : {iva_scores_pca.shape}   (S, N_PCA, T)")
print(f"IVA components (F, C)    : {iva_components.shape}    (S, N_PCA, F, C)")

---
## Step 5 — Component Ranking and Selection

`N_COMPONENTS_PCA` is typically large enough that not every IVA
component is worth visualising. We rank components by their **mean
off-diagonal correlation in `Sigma_N`** — the per-component covariance
of the source component vector (SCV) across subjects that `iva_g`
returns, normalised to a correlation matrix. Higher values indicate
components the IVA model itself judges as the most coupled across
subjects.

For each component `k`:

```
C[k]          = Sigma_N[:, :, k] normalised to a correlation matrix
rank_score[k] = mean of C[k, i, j] over i ≠ j
```

The **top `N_TOP`** and **bottom `N_BOTTOM`** components by
`rank_score` are then carried through all downstream analyses.
Showing the bottom components alongside the top ones gives a useful
noise contrast: they should look temporally / spatially inconsistent
across subjects.

| Quantity | Shape | Description |
|----------|-------|-------------|
| `sigma_corr` | `(N_PCA, S, S)` | Per-component Sigma_N normalised to a correlation matrix |
| `rank_score` | `(N_PCA,)` | Mean off-diagonal Sigma_N correlation per component |
| `TOP_INDICES` | `(N_TOP,)` | IVA component indices for the top-ranked components |
| `BOTTOM_INDICES` | `(N_BOTTOM,)` | IVA component indices for the bottom-ranked components |
| `SELECTED_INDICES` | `(N_TOP + N_BOTTOM,)` | TOP followed by BOTTOM |
| `SELECTED_RANK_LABELS` | list[str] | Display labels, e.g. `"TOP 1 (IC 7, r=+0.42)"` |
| `SELECTED_IS_TOP` | list[bool] | True for the first N_TOP entries, False after |


In [ ]:
# ``sigma_corr`` is the sign-aligned (N_PCA, K, K) correlation stack built by
# the sign-alignment cell above. Rank by mean off-diagonal correlation
# (diagonal excluded).
off_diag_mask = ~np.eye(n_subjects, dtype=bool)
rank_score = np.array(
    [sigma_corr[k][off_diag_mask].mean() for k in range(N_COMPONENTS_PCA)]
)

order = np.argsort(rank_score)[::-1]  # descending
TOP_INDICES = order[:N_TOP].tolist()
BOTTOM_INDICES = order[-N_BOTTOM:][::-1].tolist()  # worst first
SELECTED_INDICES = TOP_INDICES + BOTTOM_INDICES
SELECTED_IS_TOP = [True] * N_TOP + [False] * N_BOTTOM


def _ic_title(i: int) -> str:
    """Display label for the i-th selected component."""
    k = SELECTED_INDICES[i]
    tag = f"TOP {i + 1}" if i < N_TOP else f"BOT {i - N_TOP + 1}"
    return f"{tag} (IC {k + 1}, r={rank_score[k]:+.2f})"


SELECTED_RANK_LABELS = [_ic_title(i) for i in range(len(SELECTED_INDICES))]

print("Top components (highest mean off-diagonal Sigma_N correlation):")
for i, k in enumerate(TOP_INDICES):
    print(f"  TOP {i + 1:2d}: IC {k + 1:3d}  r = {rank_score[k]:+.3f}")
print("Bottom components (lowest mean off-diagonal Sigma_N correlation):")
for i, k in enumerate(BOTTOM_INDICES):
    print(f"  BOT {i + 1:2d}: IC {k + 1:3d}  r = {rank_score[k]:+.3f}")

# Summary bar plot: all component scores with top/bottom highlighted.
fig, ax = plt.subplots(figsize=(max(8, 0.35 * N_COMPONENTS_PCA), 4.5))
colors = ["lightgray"] * N_COMPONENTS_PCA
for k in TOP_INDICES:
    colors[k] = "steelblue"
for k in BOTTOM_INDICES:
    colors[k] = "firebrick"
xs = np.arange(N_COMPONENTS_PCA)
ax.bar(xs, rank_score, color=colors)
ax.axhline(0.0, ls="--", lw=0.6, color="gray")
ax.set_xticks(xs)
ax.set_xticklabels([f"{k + 1}" for k in range(N_COMPONENTS_PCA)], fontsize=7)
ax.set_xlabel("IVA component index")
ax.set_ylabel("Mean off-diagonal Sigma_N correlation")
ax.set_title(
    f"Component Ranking by Sigma_N Mean Off-Diagonal Correlation — {LABEL}  "
    f"(top {N_TOP} = blue, bottom {N_BOTTOM} = red)"
)
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(
        PLOTS_DIR / "iva_component_ranking.png", dpi=150, bbox_inches="tight"
    )
plt.show()
plt.close("all")


---
## Result Summary

All variables required for downstream analyses are now in memory:

| Variable | Shape | Description |
|----------|-------|-------------|
| `bb_data` | `(S, C, F, T)` | Raw 4-D wavelet power tensor |
| `bb_z` | `(S, C, F, T)` | Z-scored wavelet power tensor |
| `X_subjects` | `(S, C×F, T)` | Per-subject z-scored reshape |
| `pcas` | list[`PCA`] | Per-subject fitted PCA objects |
| `pca_evr` | `(S, N_PCA)` | Per-subject PCA explained-variance ratio |
| `X_pca` | `(N_PCA, T, S)` | IVA input (PCA-reduced) |
| `W` | `(N_PCA, N_PCA, S)` | Per-subject IVA demixing |
| `Sigma_N` | `(S, S, N_PCA)` | Per-SCV covariance across subjects |
| `sigma_corr` | `(N_PCA, S, S)` | Per-component Sigma_N normalised to a correlation matrix |
| `cost` | `(n_iter,)` | IVA convergence trace |
| `iva_scores_pca` | `(S, N_PCA, T)` | Scores in PCA subspace |
| `iva_components` | `(S, N_PCA, F, C)` | Component patterns reshaped to (freq, channel) |
| `rank_score` | `(N_PCA,)` | Mean off-diagonal Sigma_N correlation per component (ranking metric) |
| `TOP_INDICES` | `(N_TOP,)` | Component indices in the top-ranked set |
| `BOTTOM_INDICES` | `(N_BOTTOM,)` | Component indices in the bottom-ranked set |
| `SELECTED_INDICES` | `(N_TOP + N_BOTTOM,)` | TOP followed by BOTTOM |
| `SELECTED_RANK_LABELS` | list[str] | Display labels for the selected components |
| `SELECTED_IS_TOP` | list[bool] | True for top entries, False for bottom |

The kth score/component is **aligned across subjects** by
construction — no post-hoc matching is required. Downstream
notebooks can now compute ISC, topomaps, frequency profiles, etc.
on top of these variables.


# Downstream Analyses

The cells below mirror the ISC-style plots from the companion ICA
notebook
[`04-wavelet-ica-analysis/wavelet_ica_frequency_channel.ipynb`](../04-wavelet-ica-analysis/wavelet_ica_frequency_channel.ipynb),
but applied to the two natural per-component axes the IVA solution
exposes:

| Variable | Shape | Independent axis | Per-subject vector for ISC |
|----------|-------|------------------|----------------------------|
| `iva_scores_pca` | `(S, N_PCA, T)` | **time** (`T` samples) | `iva_scores_pca[s, k, :]` — length `T` |
| `iva_components` | `(S, N_PCA, F, C)` | **frequency × channel** (`F × C` features) | `iva_components[s, k].ravel()` — length `F·C` |

So in everything that follows:

- **scores ⇔ time dimension** (the kth score's temporal expression
  across subjects). This is the dimension IVA explicitly aligns the
  kth source across subjects on.
- **components ⇔ frequency × channel dimension** (the kth
  component's spatio-spectral pattern across subjects). This is the
  **independent dimension** of this notebook — the axis where the
  components live.

Components are visualised in two groups — the **top `N_TOP`** and
**bottom `N_BOTTOM`** by time-axis LOO-ISC of `iva_scores_pca` (see
Step 5 above). Most plots split into a `_top.png` and a `_bottom.png`
figure; bar charts keep both groups in one figure with a separator
line so the gap between aligned and noise-like components is visible
at a glance. All plots save under `PLOTS_DIR`
(`plots/broadband/iva_frequency_channel/`).


In [ ]:
# Pearson-r thresholds used for the cluster strip below each ISC matrix.
ISC_CLUSTER_THRESHOLDS = (0.3, 0.5, 0.7)

# Discrete colormap: light gray for singletons (0) + tab10 for groups (1..10).
_GROUP_PALETTE = list(plt.colormaps["tab10"].colors)
_CLUSTER_CMAP = ListedColormap(["#dddddd"] + _GROUP_PALETTE)
_CLUSTER_NORM = BoundaryNorm(
    np.arange(-0.5, len(_GROUP_PALETTE) + 1.5, 1.0), _CLUSTER_CMAP.N
)


def _cluster_grid(
    corr_mat: np.ndarray, thresholds=ISC_CLUSTER_THRESHOLDS
) -> np.ndarray:
    """(T, S) grid of within-row cluster IDs. Singletons → 0, groups → 1, 2, ...."""
    grid = np.zeros((len(thresholds), n_subjects), dtype=int)
    for t_idx, thr in enumerate(thresholds):
        adj = (corr_mat >= thr) & ~np.eye(n_subjects, dtype=bool)
        _, labels = connected_components(adj, directed=False)
        counts = Counter(labels.tolist())
        next_group = 1
        group_map: dict[int, int] = {}
        for s in range(n_subjects):
            lab = int(labels[s])
            if counts[lab] == 1:
                grid[t_idx, s] = 0
            else:
                if lab not in group_map:
                    group_map[lab] = next_group
                    next_group += 1
                grid[t_idx, s] = group_map[lab]
    return grid


def _plot_isc_grid(corr_per_comp, labels, fig_title, file_name):
    """Plot ISC matrices + cluster strips for a group of components."""
    n_show = len(corr_per_comp)
    fig, axes = plt.subplots(
        2,
        n_show,
        figsize=(2.6 * n_show, 5.5),
        gridspec_kw={"height_ratios": [3, 1.2]},
        constrained_layout=True,
    )
    if n_show == 1:
        axes = axes.reshape(2, 1)

    im_corr = None
    for i in range(n_show):
        corr_mat = corr_per_comp[i]
        ax_top = axes[0, i]
        im_corr = ax_top.imshow(corr_mat, vmin=-1, vmax=1, cmap="RdBu_r")
        ax_top.set_xticks(range(n_subjects))
        ax_top.set_yticks(range(n_subjects))
        ax_top.set_xticklabels([f"S{s + 1}" for s in range(n_subjects)], fontsize=7)
        ax_top.set_yticklabels([f"S{s + 1}" for s in range(n_subjects)], fontsize=7)
        ax_top.set_title(labels[i], fontsize=8)

        ax_bot = axes[1, i]
        grid = _cluster_grid(corr_mat)
        ax_bot.imshow(grid, cmap=_CLUSTER_CMAP, norm=_CLUSTER_NORM, aspect="auto")
        for ti in range(grid.shape[0]):
            for sj in range(grid.shape[1]):
                val = int(grid[ti, sj])
                if val > 0:
                    ax_bot.text(
                        sj,
                        ti,
                        str(val),
                        ha="center",
                        va="center",
                        fontsize=7,
                        color="white",
                        weight="bold",
                    )
        ax_bot.set_xticks(range(n_subjects))
        ax_bot.set_xticklabels([f"S{s + 1}" for s in range(n_subjects)], fontsize=7)
        ax_bot.set_yticks(range(len(ISC_CLUSTER_THRESHOLDS)))
        ax_bot.set_yticklabels(
            [f"r≥{thr}" for thr in ISC_CLUSTER_THRESHOLDS], fontsize=8
        )
        if i == 0:
            ax_bot.set_ylabel("Threshold")

    fig.suptitle(fig_title, fontsize=12)
    fig.colorbar(im_corr, ax=axes[0, -1], label="Pearson r", shrink=0.8)
    if SAVE_PLOTS:
        fig.savefig(PLOTS_DIR / file_name, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close("all")


---
## Analysis (a) — PCA Scree Plot

IVA uses **per-subject PCA** (cell `iva-13-pca`) to reduce the
`T` time-sample feature space before joint decomposition, so the
scree plot summarises explained variance across **all subjects**:

- **Left bar plot**: across-subject mean of
  `explained_variance_ratio_` per component, with std as error bars.
- **Right line plot**: per-subject cumulative variance (gray, thin)
  + across-subject mean cumulative (coral) with a 90 % reference.

This plot is about the **PCA dimensionality** (`N_COMPONENTS_PCA`)
chosen for IVA, not about the top/bottom ranked components — those
are component indices in the IVA output, independent of the PCA
variance ranking.

| Quantity | Shape | Description |
|----------|-------|-------------|
| `pca_evr_mean` | `(N_PCA,)` | Mean explained variance ratio across subjects |
| `pca_evr_std` | `(N_PCA,)` | Std of explained variance ratio across subjects |
| `pca_evr_cum_mean` | `(N_PCA,)` | Mean cumulative explained variance |


In [ ]:
# Across-subject summary of per-subject PCA explained-variance ratios.
pca_evr_mean = pca_evr.mean(axis=0)  # (N_PCA,)
pca_evr_std = pca_evr.std(axis=0)  # (N_PCA,)
pca_evr_cum_mean = np.cumsum(pca_evr_mean)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

xs = np.arange(1, N_COMPONENTS_PCA + 1)
axes[0].bar(xs, pca_evr_mean, yerr=pca_evr_std, color="steelblue", capsize=2)
axes[0].set_xlabel("PCA component")
axes[0].set_ylabel("Explained variance ratio")
axes[0].set_title(f"PCA Scree (mean ± std across subjects) — {LABEL}")

for k in range(n_subjects):
    axes[1].plot(xs, np.cumsum(pca_evr[k]), lw=0.6, alpha=0.5, color="gray")
axes[1].plot(xs, pca_evr_cum_mean, "o-", color="coral", label="mean cumulative")
axes[1].axhline(0.9, ls="--", color="gray", label="90%")
axes[1].set_xlabel("Number of components")
axes[1].set_ylabel("Cumulative variance explained")
axes[1].set_title(f"Cumulative Variance — {LABEL}")
axes[1].legend()

fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "pca_scree.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close("all")

print(
    f"All {N_COMPONENTS_PCA} PCA components explain "
    f"{pca_evr_cum_mean[-1] * 100:.1f}% of variance on average."
)


---
## Analysis (b) — Intersubject Correlation of Score Timecourses (Time Dimension)

**Scores ⇔ time.** For each IVA component the per-subject vector is
the length-`T` row `iva_scores_pca[s, k, :]` — the kth score's
temporal expression for subject `s`. We build a subject × subject
Pearson correlation matrix per component over this time axis.
Because IVA aligns the kth score across subjects by construction,
high off-diagonal correlations reflect a temporally consistent
activation pattern.

A **cluster strip** beneath each correlation matrix colors subjects
by connected-component membership at each ISC threshold (`r ≥ 0.3,
0.5, 0.7`). Singletons stay gray; numeric group IDs disambiguate
clusters within a row. This mirrors the cluster strip used in
`04-wavelet-ica-analysis/wavelet_ica_frequency_channel.ipynb`.

| Quantity | Shape | Description |
|----------|-------|-------------|
| `score_corr_per_comp` | `(N_SHOW, S, S)` | Per-IC subject × subject score correlation along the time axis |

In [ ]:
# Subject × subject correlations of score timecourses, per component.
# Computed separately for top and bottom groups so figures stay readable.
score_corr_top = np.stack(
    [np.corrcoef(iva_scores_pca[:, k, :]) for k in TOP_INDICES]
)  # (N_TOP, S, S)
score_corr_bot = np.stack(
    [np.corrcoef(iva_scores_pca[:, k, :]) for k in BOTTOM_INDICES]
)  # (N_BOTTOM, S, S)

_plot_isc_grid(
    score_corr_top,
    labels=SELECTED_RANK_LABELS[:N_TOP],
    fig_title=(
        f"Intersubject Correlation of IVA Score Timecourses — "
        f"TOP {N_TOP} — {LABEL}"
    ),
    file_name="iva_score_isc_time_top.png",
)
_plot_isc_grid(
    score_corr_bot,
    labels=SELECTED_RANK_LABELS[N_TOP:],
    fig_title=(
        f"Intersubject Correlation of IVA Score Timecourses — "
        f"BOTTOM {N_BOTTOM} — {LABEL}"
    ),
    file_name="iva_score_isc_time_bottom.png",
)


---
## Analysis (c) — Intersubject Correlation of Component Patterns (Frequency × Channel Dimension)

**Components ⇔ frequency × channel.** Same construction as (b), but
the per-subject vector for component `k` is now the flattened
`(F × C)` pattern `iva_components[s, k].ravel()` — the kth
component's spatio-spectral fingerprint for subject `s`, which is
the **independent dimension** of this notebook. Pearson correlations
across subjects quantify whether the kth component lands on similar
`(frequency, channel)` loadings across individuals; this is
complementary to the score-timecourse alignment that IVA enforces by
construction in (b).

| Quantity | Shape | Description |
|----------|-------|-------------|
| `pattern_corr_per_comp` | `(N_SHOW, S, S)` | Per-IC subject × subject pattern correlation along the (F × C) axis |

In [ ]:
# Subject × subject correlations of (F, C) component patterns, per IC.
# Each subject's pattern is iva_components[s, k] of shape (F, C),
# flattened to a length-(F*C) vector so the correlation is taken over the
# combined frequency-channel feature axis (the independent dimension).
def _pattern_corr(indices):
    out = np.zeros((len(indices), n_subjects, n_subjects))
    for i, k in enumerate(indices):
        flat = iva_components[:, k, :, :].reshape(n_subjects, -1)  # (S, F*C)
        out[i] = np.corrcoef(flat)
    return out


pattern_corr_top = _pattern_corr(TOP_INDICES)
pattern_corr_bot = _pattern_corr(BOTTOM_INDICES)

_plot_isc_grid(
    pattern_corr_top,
    labels=SELECTED_RANK_LABELS[:N_TOP],
    fig_title=(
        f"Intersubject Correlation of IVA Component (F × C) Patterns — "
        f"TOP {N_TOP} — {LABEL}"
    ),
    file_name="iva_pattern_isc_freq_channel_top.png",
)
_plot_isc_grid(
    pattern_corr_bot,
    labels=SELECTED_RANK_LABELS[N_TOP:],
    fig_title=(
        f"Intersubject Correlation of IVA Component (F × C) Patterns — "
        f"BOTTOM {N_BOTTOM} — {LABEL}"
    ),
    file_name="iva_pattern_isc_freq_channel_bottom.png",
)


---
## Analysis (d) — Mean LOO-ISC per Component (Time Dimension)

Whole-recording leave-one-out inter-subject correlation per IVA
component over the **time** axis — i.e. computed from the per-subject
score timecourses `iva_scores_pca[s, k, :]`. For each component `k`:

```
for each subject s:
    others_mean = mean over s' ≠ s of iva_scores_pca[s', k, :]
    r[k, s]     = pearson(iva_scores_pca[s, k, :], others_mean)
```

The bar height is the across-subject **mean** of `r[k, s]`; the error
bar is the across-subject **std**. Bars are red where the mean is
negative. Pair this with (e) below for the matching summary along
the frequency × channel axis.

| Quantity | Shape | Description |
|----------|-------|-------------|
| `loo_isc_time_per_subject` | `(N_SHOW, S)` | Whole-recording LOO-ISC per IC and subject (time axis) |
| `loo_isc_time_mean` | `(N_SHOW,)` | Across-subject mean LOO-ISC per IC (time axis) |
| `loo_isc_time_std` | `(N_SHOW,)` | Across-subject std LOO-ISC per IC (time axis) |

In [ ]:
# Time-dimension LOO-ISC: per-subject vector = score timecourse
# iva_scores_pca[s, k, :]. Bars cover top + bottom in one figure so the
# gap between aligned and noise-like components is visible at a glance.
loo_isc_time_per_subject = np.zeros((N_COMPONENTS_SHOW, n_subjects))
for i, k in enumerate(SELECTED_INDICES):
    subj_vectors = iva_scores_pca[:, k, :]  # (S, T)
    for s in range(n_subjects):
        others_mean = np.delete(subj_vectors, s, axis=0).mean(axis=0)
        loo_isc_time_per_subject[i, s] = float(
            pearsonr(subj_vectors[s], others_mean)[0]
        )

loo_isc_time_mean = loo_isc_time_per_subject.mean(axis=1)
loo_isc_time_std = loo_isc_time_per_subject.std(axis=1)

bar_colors = []
for i, m in enumerate(loo_isc_time_mean):
    if SELECTED_IS_TOP[i]:
        bar_colors.append("firebrick" if m < 0 else "steelblue")
    else:
        bar_colors.append("salmon" if m < 0 else "lightsteelblue")

fig, ax = plt.subplots(figsize=(max(10, 0.9 * N_COMPONENTS_SHOW), 5.0))
xs = np.arange(N_COMPONENTS_SHOW)
ax.bar(xs, loo_isc_time_mean, yerr=loo_isc_time_std, color=bar_colors, capsize=4)
ax.axhline(0.0, ls="--", lw=0.6, color="gray")
ax.axvline(
    N_TOP - 0.5,
    ls="--",
    lw=0.9,
    color="black",
    alpha=0.5,
    label=f"top {N_TOP} | bottom {N_BOTTOM}",
)
ax.set_xticks(xs)
ax.set_xticklabels(SELECTED_RANK_LABELS, fontsize=7, rotation=45, ha="right")
ax.set_xlabel("Component")
ax.set_ylabel("Mean LOO-ISC across subjects")
ax.set_ylim(-1.05, 1.05)
ax.set_title(
    f"Per-IC Mean LOO-ISC — Time Dimension (score timecourses) — {LABEL}"
)
ax.legend(loc="upper right", fontsize=8)

fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "iva_loo_isc_bar_time.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close("all")


---
## Analysis (e) — Mean LOO-ISC per Component (Frequency × Channel Dimension)

Companion to (d) along the **independent dimension** of this
notebook. Whole-recording LOO-ISC per IVA component using the
per-subject component pattern flattened over `(F × C)`. For each
component `k`:

```
for each subject s:
    pat[s']     = iva_components[s', k, :, :].ravel()  # length F*C
    others_mean = mean over s' ≠ s of pat[s']
    r[k, s]     = pearson(pat[s], others_mean)
```

Same bar / errorbar / sign-coloring conventions as (d), so the two
plots can be compared side by side: (d) shows how strongly each
component aligns **across time**, (e) shows how strongly each
component aligns **across frequency × channel loadings**.

| Quantity | Shape | Description |
|----------|-------|-------------|
| `loo_isc_fc_per_subject` | `(N_SHOW, S)` | Whole-recording LOO-ISC per IC and subject (F × C axis) |
| `loo_isc_fc_mean` | `(N_SHOW,)` | Across-subject mean LOO-ISC per IC (F × C axis) |
| `loo_isc_fc_std` | `(N_SHOW,)` | Across-subject std LOO-ISC per IC (F × C axis) |

In [ ]:
# (F × C)-dimension LOO-ISC: per-subject vector = flattened pattern
# iva_components[s, k].ravel() (length F*C). Same plot layout as (d).
loo_isc_fc_per_subject = np.zeros((N_COMPONENTS_SHOW, n_subjects))
for i, k in enumerate(SELECTED_INDICES):
    subj_vectors = iva_components[:, k, :, :].reshape(n_subjects, -1)  # (S, F*C)
    for s in range(n_subjects):
        others_mean = np.delete(subj_vectors, s, axis=0).mean(axis=0)
        loo_isc_fc_per_subject[i, s] = float(
            pearsonr(subj_vectors[s], others_mean)[0]
        )

loo_isc_fc_mean = loo_isc_fc_per_subject.mean(axis=1)
loo_isc_fc_std = loo_isc_fc_per_subject.std(axis=1)

bar_colors = []
for i, m in enumerate(loo_isc_fc_mean):
    if SELECTED_IS_TOP[i]:
        bar_colors.append("firebrick" if m < 0 else "steelblue")
    else:
        bar_colors.append("salmon" if m < 0 else "lightsteelblue")

fig, ax = plt.subplots(figsize=(max(10, 0.9 * N_COMPONENTS_SHOW), 5.0))
xs = np.arange(N_COMPONENTS_SHOW)
ax.bar(xs, loo_isc_fc_mean, yerr=loo_isc_fc_std, color=bar_colors, capsize=4)
ax.axhline(0.0, ls="--", lw=0.6, color="gray")
ax.axvline(
    N_TOP - 0.5,
    ls="--",
    lw=0.9,
    color="black",
    alpha=0.5,
    label=f"top {N_TOP} | bottom {N_BOTTOM}",
)
ax.set_xticks(xs)
ax.set_xticklabels(SELECTED_RANK_LABELS, fontsize=7, rotation=45, ha="right")
ax.set_xlabel("Component")
ax.set_ylabel("Mean LOO-ISC across subjects")
ax.set_ylim(-1.05, 1.05)
ax.set_title(
    f"Per-IC Mean LOO-ISC — Frequency × Channel Dimension "
    f"(component patterns) — {LABEL}"
)
ax.legend(loc="upper right", fontsize=8)

fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(
        PLOTS_DIR / "iva_loo_isc_bar_freq_channel.png",
        dpi=150,
        bbox_inches="tight",
    )
plt.show()
plt.close("all")


---
## Analysis (f) — Topomap of Mean and Variance Across Subjects

Computed from **`iva_components`** (the scores have no channel axis,
so this is the only way to project the kth component onto the
scalp). For every component `k` each subject has their own
`(F × C)` pattern; collapsing the frequency axis yields a
per-subject **channel loading** vector:

```
chan_loading[s, k, c] = mean over f of iva_components[s, k, f, c]
```

**Each subject's map is rescaled to unit L2 norm before averaging.**
`iva_g` fixes the *source* scale (its unmixing rows are unit-norm in the
whitened space) but not the *pattern* scale, so the per-subject maps differ by
a subject-specific gain. Unnormalised, the largest-gain subjects dominate the
mean map and their amplitude — rather than genuine topographic spread — drives
the variance map. `normalize_patterns_per_subject` removes exactly that gain
and nothing else (the direction of every pattern is untouched).

We then summarise the across-subject distribution per channel with
the **mean** and the **variance**, giving two topomaps per
component: the mean spatial fingerprint (`RdBu_r`, symmetric) and
the across-subject dispersion (`viridis`, non-negative).

| Quantity | Shape | Description |
|----------|-------|-------------|
| `chan_loading` | `(S, N_PCA, C)` | Per-subject channel loading per IC, unit L2 norm |
| `chan_mean` | `(N_PCA, C)` | Mean channel loading across subjects |
| `chan_var` | `(N_PCA, C)` | Variance of channel loading across subjects |

In [ ]:
# Per-subject channel loading (collapse frequency axis): (S, N_PCA, C)
# Then rescale each subject's map to unit L2 norm before the across-subject
# summaries — IVA leaves a per-subject gain on the patterns, which would
# otherwise dominate the mean and leak into the variance map.
chan_loading = normalize_patterns_per_subject(iva_components.mean(axis=2))
chan_mean = chan_loading.mean(axis=0)  # (N_PCA, C)
chan_var = chan_loading.var(axis=0)  # (N_PCA, C)

# MNE info for the topomap, restricted to the channel subset.
info = analyzers[LABEL].info
info = mne.pick_info(info, mne.pick_types(info, eeg=True))
if n_channels < len(info.ch_names):
    info = mne.pick_info(info, list(range(n_channels)))


def _plot_topomap_group(indices, labels, group_name, file_name):
    n_show = len(indices)
    fig, axes = plt.subplots(2, n_show, figsize=(3.0 * n_show, 7.0))
    if n_show == 1:
        axes = axes.reshape(2, 1)
    for i, k in enumerate(indices):
        vlim_m = float(np.percentile(np.abs(chan_mean[k]), 99))
        if vlim_m == 0.0:
            vlim_m = 1e-12
        im_m, _ = plot_topomap(
            chan_mean[k], info, axes=axes[0, i], show=False,
            cmap="RdBu_r", vlim=(-vlim_m, vlim_m),
        )
        axes[0, i].set_title(labels[i], fontsize=8)
        fig.colorbar(im_m, ax=axes[0, i], fraction=0.046, pad=0.04)

        vlim_v = float(np.percentile(chan_var[k], 99))
        if vlim_v == 0.0:
            vlim_v = 1e-12
        im_v, _ = plot_topomap(
            chan_var[k], info, axes=axes[1, i], show=False,
            cmap="viridis", vlim=(0.0, vlim_v),
        )
        fig.colorbar(im_v, ax=axes[1, i], fraction=0.046, pad=0.04)

    fig.text(
        0.01, 0.75, "Mean across subjects (unit-norm)", rotation=90, va="center",
        fontsize=11, fontweight="bold",
    )
    fig.text(
        0.01, 0.25, "Variance across subjects (unit-norm)", rotation=90, va="center",
        fontsize=11, fontweight="bold",
    )
    fig.suptitle(
        f"Mean and Variance Topomaps Across Subjects (unit-norm patterns) — "
        f"{group_name} — {LABEL}",
        fontsize=13,
    )
    fig.tight_layout(rect=(0.03, 0, 1, 0.97))
    if SAVE_PLOTS:
        fig.savefig(PLOTS_DIR / file_name, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close("all")


_plot_topomap_group(
    TOP_INDICES, SELECTED_RANK_LABELS[:N_TOP],
    group_name=f"TOP {N_TOP}", file_name="iva_topomap_mean_var_top.png",
)
_plot_topomap_group(
    BOTTOM_INDICES, SELECTED_RANK_LABELS[N_TOP:],
    group_name=f"BOTTOM {N_BOTTOM}", file_name="iva_topomap_mean_var_bottom.png",
)

---
## Analysis (g) — Time × Frequency Map per Component (Mean Across Subjects)

Combines both ingredients we have: the **frequency profile** is the
subject-averaged, channel-averaged loading from `iva_components`,
and the **time profile** is the subject-averaged source timecourse
from `iva_scores_pca`. Their outer product gives a rank-1
approximation of the component's `(F, T)` expression that is
grounded entirely in the IVA solution — channels are averaged out
of the patterns, and subjects are averaged out of both axes.

```
freq_profile[k] = iva_components.mean(axis=0)[k].mean(axis=1)   # (F,) channel-avg
time_profile[k] = iva_scores_pca.mean(axis=0)[k]                # (T,) subject-avg
tf_map[k]       = outer(freq_profile[k], time_profile[k])       # (F, T)
```

| Quantity | Shape | Description |
|----------|-------|-------------|
| `freq_profiles` | `(F, N_PCA)` | Channel-averaged loading per frequency, per IC (mean across subjects) |
| `time_profiles` | `(N_PCA, T)` | Subject-averaged score timecourse per IC |
| `ft_maps` | `(N_PCA, F, T)` | Outer-product frequency × time map per component |

In [ ]:
# Subject-averaged spatial-spectral pattern: (N_PCA, F, C)
mean_components = iva_components.mean(axis=0)
# Channel-averaged frequency profile, transposed to (F, N_PCA) for einsum
freq_profiles = mean_components.mean(axis=2).T  # (F, N_PCA)
# Subject-averaged score timecourse: (N_PCA, T)
time_profiles = iva_scores_pca.mean(axis=0)

# Outer product per component → (N_PCA, F, T)
ft_maps = np.einsum("fk,kt->kft", freq_profiles, time_profiles)


def _plot_tf_group(indices, labels, group_name, file_name):
    n_show = len(indices)
    fig, axes = plt.subplots(n_show, 1, figsize=(14, 2.6 * n_show), sharex=True)
    if n_show == 1:
        axes = [axes]
    for ax, lbl, k in zip(axes, labels, indices):
        data_i = ft_maps[k]
        vlim_i = max(float(np.percentile(np.abs(data_i), 99)), 1e-12)
        mesh = ax.pcolormesh(
            time, FREQS, data_i, cmap="RdBu_r",
            vmin=-vlim_i, vmax=vlim_i, shading="auto",
        )
        ax.set_ylabel("Freq (Hz)")
        ax.set_title(f"{lbl} — Time × Frequency (outer product)", fontsize=10)
        fig.colorbar(mesh, ax=ax, pad=0.01, fraction=0.025)
    axes[-1].set_xlabel("Time (s)")
    fig.suptitle(
        f"Per-IC Time × Frequency Maps (mean across subjects) — "
        f"{group_name} — {LABEL}",
        fontsize=13,
        y=1.01,
    )
    fig.tight_layout()
    if SAVE_PLOTS:
        fig.savefig(PLOTS_DIR / file_name, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close("all")


_plot_tf_group(
    TOP_INDICES, SELECTED_RANK_LABELS[:N_TOP],
    group_name=f"TOP {N_TOP}", file_name="iva_time_frequency_top.png",
)
_plot_tf_group(
    BOTTOM_INDICES, SELECTED_RANK_LABELS[N_TOP:],
    group_name=f"BOTTOM {N_BOTTOM}", file_name="iva_time_frequency_bottom.png",
)


---
## Analysis (h) — Score Timecourses: Mean and Variance Across Subjects

Per-component **temporal resolution**: for each IVA component the
per-subject score `iva_scores_pca[s, k, :]` is collapsed across the
subject axis at every time point:

```
mean_temporal[k, t] = mean over s of iva_scores_pca[s, k, t]   # (K, T)
var_temporal[k, t]  = var  over s of iva_scores_pca[s, k, t]   # (K, T)
std_temporal[k, t]  = sqrt(var_temporal[k, t])                 # (K, T)
```

The **mean** is plotted as a line and the **variance** as an
error-shading band (`mean ± √variance`) around it. Narrow bands
mark time points where subjects agree on the kth component's
activation; wide bands mark idiosyncratic intervals.

| Quantity | Shape | Description |
|----------|-------|-------------|
| `mean_temporal` | `(N_PCA, T)` | Mean score across subjects |
| `var_temporal` | `(N_PCA, T)` | Variance of score across subjects |
| `std_temporal` | `(N_PCA, T)` | Std (band width) |

In [ ]:
# Across-subject summaries of each IVA component's score timecourse.
mean_temporal = iva_scores_pca.mean(axis=0)  # (N_PCA, T)
var_temporal = iva_scores_pca.var(axis=0)  # (N_PCA, T)
std_temporal = np.sqrt(var_temporal)  # used as error-band width


def _plot_mean_var_group(indices, labels, group_name, file_name):
    n_show = len(indices)
    fig, axes = plt.subplots(n_show, 1, figsize=(14, 2.5 * n_show), sharex=True)
    if n_show == 1:
        axes = [axes]
    for i, (ax, lbl, k) in enumerate(zip(axes, labels, indices)):
        ax.plot(time, mean_temporal[k], lw=0.9, color="darkorange", label="mean")
        ax.fill_between(
            time,
            mean_temporal[k] - std_temporal[k],
            mean_temporal[k] + std_temporal[k],
            alpha=0.25,
            color="darkorange",
            label="± √variance",
        )
        ax.set_ylabel(lbl, fontsize=8)
        ax.set_title(f"{lbl} — Mean & Variance Across Subjects", fontsize=10)
        if i == 0:
            ax.legend(loc="upper right", fontsize=8)
    axes[-1].set_xlabel("Time (s)")
    fig.suptitle(
        f"Per-IC Score Timecourse — Mean and Variance — "
        f"{group_name} — {LABEL}",
        fontsize=13,
        y=1.01,
    )
    fig.tight_layout()
    if SAVE_PLOTS:
        fig.savefig(PLOTS_DIR / file_name, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close("all")


_plot_mean_var_group(
    TOP_INDICES, SELECTED_RANK_LABELS[:N_TOP],
    group_name=f"TOP {N_TOP}",
    file_name="iva_mean_variance_over_time_top.png",
)
_plot_mean_var_group(
    BOTTOM_INDICES, SELECTED_RANK_LABELS[N_TOP:],
    group_name=f"BOTTOM {N_BOTTOM}",
    file_name="iva_mean_variance_over_time_bottom.png",
)


---
## Analysis (i) — Pair-Space Heatmaps per Component

For every component `k` we show four `(X × Y)` heatmaps — **first
name in the title is the x-axis, second is the y-axis**, following
the 04 ICA notebook convention. The axis priority is:

1. **time** is on x whenever it's in the pair (matches 04 ICA cell 31, "Time × Subject");
2. if there's no time, **subject** is on x;
3. if there's neither time nor subject, **frequency** is on x (matches 04 ICA cell 33, "Frequency × Channel").

The third dimension is collapsed by averaging.

| Variant | Source | Collapsed axis | x | y |
|---------|--------|----------------|----|----|
| **Time × Subject** | `iva_scores_pca[:, k, :]` | — (scores have no F/C) | time | subject |
| **Subject × Frequency** | `iva_components[:, k, :, :].mean(axis=-1)` | channel | subject | frequency |
| **Subject × Channel** | `iva_components[:, k, :, :].mean(axis=-2)` | frequency | subject | channel |
| **Frequency × Channel** | `iva_components[:, k, :, :].mean(axis=0)` | subject | frequency | channel |

A diverging colormap (`RdBu_r`, symmetric around zero) preserves
the sign in every panel. Color limits are **shared across
components** for the three subject-involving variants (so
components are visually comparable on that variant — matches 04
ICA cell 31), and **per-component** for the Frequency × Channel
variant so weaker components are not flattened by stronger ones
(matches 04 ICA cell 33).

| Quantity | Shape | Description |
|----------|-------|-------------|
| `time_subj` | `(N_SHOW, S, T)` | Per-IC matrices for Time × Subject (rows = subject = y, cols = time = x) |
| `subj_freq` | `(N_SHOW, F, S)` | Per-IC matrices for Subject × Frequency (rows = frequency = y, cols = subject = x) |
| `subj_chan` | `(N_SHOW, C, S)` | Per-IC matrices for Subject × Channel (rows = channel = y, cols = subject = x) |
| `freq_chan` | `(N_SHOW, F, C)` | Per-IC subject-averaged `(F, C)` patterns; transposed to `(C, F)` at plot time |

In [ ]:
channels = np.arange(n_channels)
subjects_idx = np.arange(n_subjects)
subject_labels = [f"S{s + 1}" for s in subjects_idx]


def _safe_vlim(arr: np.ndarray) -> float:
    """99th percentile of |arr|, never zero (so vmin/vmax stay valid)."""
    return max(float(np.percentile(np.abs(arr), 99)), 1e-12)


def _plot_time_subject_group(indices, labels, group_name, file_name):
    # x=time, y=subject. Source: iva_scores_pca (scores live in time).
    data = iva_scores_pca.transpose(1, 0, 2)  # (K, S, T)
    sel = data[indices]
    vlim = _safe_vlim(sel)
    n_show = len(indices)
    fig, axes = plt.subplots(n_show, 1, figsize=(14, 2.6 * n_show), sharex=True)
    if n_show == 1:
        axes = [axes]
    for ax, lbl, mat in zip(axes, labels, sel):
        mesh = ax.pcolormesh(
            time, subjects_idx, mat, cmap="RdBu_r",
            vmin=-vlim, vmax=vlim, shading="auto",
        )
        ax.set_yticks(subjects_idx)
        ax.set_yticklabels(subject_labels, fontsize=8)
        ax.set_ylabel("Subject")
        ax.set_title(f"{lbl} — Time × Subject", fontsize=10)
        fig.colorbar(mesh, ax=ax, pad=0.01, fraction=0.025, label="score")
    axes[-1].set_xlabel("Time (s)")
    fig.suptitle(
        f"Time × Subject per IC (from scores) — {group_name} — {LABEL}",
        fontsize=13, y=1.01,
    )
    fig.tight_layout()
    if SAVE_PLOTS:
        fig.savefig(PLOTS_DIR / file_name, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close("all")


def _plot_subject_freq_group(indices, labels, group_name, file_name):
    # x=subject, y=frequency. Channel-avg components.
    data = iva_components.mean(axis=-1).transpose(1, 2, 0)  # (K, F, S)
    sel = data[indices]
    vlim = _safe_vlim(sel)
    n_show = len(indices)
    fig, axes = plt.subplots(1, n_show, figsize=(3.5 * n_show, 5.0), sharey=True)
    if n_show == 1:
        axes = [axes]
    for ax, lbl, mat in zip(axes, labels, sel):
        mesh = ax.pcolormesh(
            subjects_idx, FREQS, mat, cmap="RdBu_r",
            vmin=-vlim, vmax=vlim, shading="auto",
        )
        ax.set_xticks(subjects_idx)
        ax.set_xticklabels(subject_labels, fontsize=8)
        ax.set_xlabel("Subject")
        ax.set_title(lbl, fontsize=8)
        fig.colorbar(mesh, ax=ax, fraction=0.046, pad=0.04, label="loading")
    axes[0].set_ylabel("Frequency (Hz)")
    fig.suptitle(
        f"Subject × Frequency per IC (channel-avg components) — "
        f"{group_name} — {LABEL}",
        fontsize=13, y=1.02,
    )
    fig.tight_layout()
    if SAVE_PLOTS:
        fig.savefig(PLOTS_DIR / file_name, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close("all")


def _plot_subject_chan_group(indices, labels, group_name, file_name):
    # x=subject, y=channel. Frequency-avg components.
    data = iva_components.mean(axis=-2).transpose(1, 2, 0)  # (K, C, S)
    sel = data[indices]
    vlim = _safe_vlim(sel)
    n_show = len(indices)
    fig, axes = plt.subplots(1, n_show, figsize=(3.5 * n_show, 5.0), sharey=True)
    if n_show == 1:
        axes = [axes]
    for ax, lbl, mat in zip(axes, labels, sel):
        mesh = ax.pcolormesh(
            subjects_idx, channels, mat, cmap="RdBu_r",
            vmin=-vlim, vmax=vlim, shading="auto",
        )
        ax.set_xticks(subjects_idx)
        ax.set_xticklabels(subject_labels, fontsize=8)
        ax.set_xlabel("Subject")
        ax.set_title(lbl, fontsize=8)
        fig.colorbar(mesh, ax=ax, fraction=0.046, pad=0.04, label="loading")
    axes[0].set_ylabel("Channel")
    fig.suptitle(
        f"Subject × Channel per IC (frequency-avg components) — "
        f"{group_name} — {LABEL}",
        fontsize=13, y=1.02,
    )
    fig.tight_layout()
    if SAVE_PLOTS:
        fig.savefig(PLOTS_DIR / file_name, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close("all")


def _plot_freq_chan_group(indices, labels, group_name, file_name):
    # x=frequency, y=channel. Subject-averaged components. Per-IC vlim.
    data = iva_components.mean(axis=0)  # (K, F, C)
    sel = data[indices]
    n_show = len(indices)
    fig, axes = plt.subplots(1, n_show, figsize=(3.5 * n_show, 4.5), sharey=True)
    if n_show == 1:
        axes = [axes]
    for ax, lbl, mat in zip(axes, labels, sel):
        data_i = mat.T  # (C, F)
        vlim_i = _safe_vlim(data_i)
        mesh = ax.pcolormesh(
            FREQS, channels, data_i, cmap="RdBu_r",
            vmin=-vlim_i, vmax=vlim_i, shading="auto",
        )
        ax.set_xlabel("Frequency (Hz)")
        ax.set_title(lbl, fontsize=8)
        fig.colorbar(mesh, ax=ax, fraction=0.046, pad=0.04, label="loading")
    axes[0].set_ylabel("Channel")
    fig.suptitle(
        f"Frequency × Channel per IC (subject-avg components) — "
        f"{group_name} — {LABEL}",
        fontsize=13, y=1.02,
    )
    fig.tight_layout()
    if SAVE_PLOTS:
        fig.savefig(PLOTS_DIR / file_name, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close("all")


# ── (1) Time × Subject — Matches 04 ICA cell 31. ──────
_plot_time_subject_group(
    TOP_INDICES, SELECTED_RANK_LABELS[:N_TOP],
    group_name=f"TOP {N_TOP}",
    file_name="iva_pairmap_time_subject_top.png",
)
_plot_time_subject_group(
    BOTTOM_INDICES, SELECTED_RANK_LABELS[N_TOP:],
    group_name=f"BOTTOM {N_BOTTOM}",
    file_name="iva_pairmap_time_subject_bottom.png",
)

# ── (2) Subject × Frequency ──────
_plot_subject_freq_group(
    TOP_INDICES, SELECTED_RANK_LABELS[:N_TOP],
    group_name=f"TOP {N_TOP}",
    file_name="iva_pairmap_subject_frequency_top.png",
)
_plot_subject_freq_group(
    BOTTOM_INDICES, SELECTED_RANK_LABELS[N_TOP:],
    group_name=f"BOTTOM {N_BOTTOM}",
    file_name="iva_pairmap_subject_frequency_bottom.png",
)

# ── (3) Subject × Channel ──────
_plot_subject_chan_group(
    TOP_INDICES, SELECTED_RANK_LABELS[:N_TOP],
    group_name=f"TOP {N_TOP}",
    file_name="iva_pairmap_subject_channel_top.png",
)
_plot_subject_chan_group(
    BOTTOM_INDICES, SELECTED_RANK_LABELS[N_TOP:],
    group_name=f"BOTTOM {N_BOTTOM}",
    file_name="iva_pairmap_subject_channel_bottom.png",
)

# ── (4) Frequency × Channel — Matches 04 ICA cell 33. ──────
_plot_freq_chan_group(
    TOP_INDICES, SELECTED_RANK_LABELS[:N_TOP],
    group_name=f"TOP {N_TOP}",
    file_name="iva_pairmap_frequency_channel_top.png",
)
_plot_freq_chan_group(
    BOTTOM_INDICES, SELECTED_RANK_LABELS[N_TOP:],
    group_name=f"BOTTOM {N_BOTTOM}",
    file_name="iva_pairmap_frequency_channel_bottom.png",
)


---
## Analysis (j) — Mean Subject Loading per Component

For each IVA component the **mean absolute score over time** per
subject — a scalar per `(subject, component)` pair summarising how
strongly each participant expresses the kth source across the
recording:

```
subject_loadings[s, k] = mean over t of |iva_scores_pca[s, k, t]|
```

Subjects with uniformly high loadings indicate a stimulus-driven
mode; uneven loadings reflect individual differences. Mirrors
04 ICA cell 21 (the same calculation against `scores_3d`).

| Quantity | Shape | Description |
|----------|-------|-------------|
| `subject_loadings` | `(S, N_PCA)` | Mean `|score|` over time, per subject and component |

In [ ]:
# Per-subject mean |score| over time: (S, K)
subject_loadings = np.abs(iva_scores_pca).mean(axis=2)  # (S, K)


def _plot_loadings_group(indices, labels, group_name, file_name):
    n_show = len(indices)
    fig, axes = plt.subplots(1, n_show, figsize=(3 * n_show, 4), sharey=True)
    if n_show == 1:
        axes = [axes]
    for ax, lbl, k in zip(axes, labels, indices):
        ax.barh(
            range(n_subjects),
            subject_loadings[:, k],
            color="darkorange",
        )
        ax.set_yticks(range(n_subjects))
        ax.set_yticklabels([f"S{s + 1}" for s in range(n_subjects)], fontsize=8)
        ax.set_xlabel("|score|")
        ax.set_title(lbl, fontsize=8)
    axes[0].set_ylabel("Subject")
    fig.suptitle(
        f"Per-Subject Mean Loading per Component — {group_name} — {LABEL}",
        fontsize=13, y=1.02,
    )
    fig.tight_layout()
    if SAVE_PLOTS:
        fig.savefig(PLOTS_DIR / file_name, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close("all")


_plot_loadings_group(
    TOP_INDICES, SELECTED_RANK_LABELS[:N_TOP],
    group_name=f"TOP {N_TOP}",
    file_name="iva_subject_loadings_top.png",
)
_plot_loadings_group(
    BOTTOM_INDICES, SELECTED_RANK_LABELS[N_TOP:],
    group_name=f"BOTTOM {N_BOTTOM}",
    file_name="iva_subject_loadings_bottom.png",
)


---
## Analysis (k) — Subject × Subject Correlation from Sigma_N

`iva_g` returns **`Sigma_N`** of shape `(S, S, N_PCA)` — the per-component
covariance of the source component vector (SCV) across subjects, estimated
by IVA itself. Step 5 normalises each `(S, S)` slice to a correlation
matrix (`sigma_corr`) and uses its mean off-diagonal value as the global
ranking metric.

This cell plots `sigma_corr` for the same `SELECTED_INDICES` shown in
(b)/(c), with the same `_plot_isc_grid` helper. Because these are the
exact values that drive `rank_score`, the plot is the natural visual
companion to the Step 5 bar chart — and a direct contrast to (b), which
computes Pearson-r post-hoc from `iva_scores_pca`.


In [ ]:
# Plot the Sigma_N-derived subject × subject correlation matrix for the
# globally selected components (Step 5). `sigma_corr` and the indices
# come from Step 5 — no re-ranking here.
sigma_corr_top = np.stack([sigma_corr[k] for k in TOP_INDICES])
sigma_corr_bot = np.stack([sigma_corr[k] for k in BOTTOM_INDICES])

_plot_isc_grid(
    sigma_corr_top,
    labels=SELECTED_RANK_LABELS[:N_TOP],
    fig_title=(
        f"Subject × Subject Correlation from Sigma_N — "
        f"TOP {N_TOP} — {LABEL}"
    ),
    file_name="iva_sigma_n_corr_top.png",
)
_plot_isc_grid(
    sigma_corr_bot,
    labels=SELECTED_RANK_LABELS[N_TOP:],
    fig_title=(
        f"Subject × Subject Correlation from Sigma_N — "
        f"BOTTOM {N_BOTTOM} — {LABEL}"
    ),
    file_name="iva_sigma_n_corr_bottom.png",
)
